# PCR — Knowledge Distillation + LoRA + Random Negatives


In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc
import json
import math
import random
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPUs:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))


PyTorch: 2.11.0+cu130
CUDA: True
GPUs: 2
0 Tesla T4
1 Tesla T4


In [2]:
# -------------------------
# Configuration
# -------------------------
SEED = 42

MAX_SEQ_LEN = 1024
CHUNK_OVERLAP = 64
CHUNK_CONTENT_LEN = MAX_SEQ_LEN - 2
CHUNK_STRIDE = CHUNK_CONTENT_LEN - CHUNK_OVERLAP
MAX_CHUNKS = 8

# Fixed PCR settings
CHUNK_BATCH_SIZE = 8
NUM_NEGATIVES = 4
BATCH_SIZE = 8

TEMPERATURE = 0.07
KD_WEIGHT = 0.5
EPOCHS = 10
GRAD_CLIP = 1.0
WEIGHT_DECAY = 0.01

# Standard additive LoRA
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_A_LR = 2e-5
LORA_B_LR = 2e-5

# KD student depth
TEACHER_N_LAYERS = 12
STUDENT_N_LAYERS = 6

USE_GRAD_CHECKPOINTING = True
USE_AMP = True
EVAL_DOCUMENT_BATCH_SIZE = 1

TOKENIZER_PATH = "/kaggle/input/datasets/rogue2222/tokenizer-results/project_bpe_16k.pkl"

# Existing best PCR LoRA checkpoint becomes the FROZEN teacher.
TEACHER_CHECKPOINT_PATH = "/kaggle/input/datasets/rogue2222/kerenel-p7/PCR_LinearAttention_LoRA_Random_Negatives/best.pt"

# Only used as a fallback when a teacher checkpoint is unavailable.
BASE_CHECKPOINT_PATH = "/kaggle/input/datasets/saiveeru341251/decoder1bmodel/rr_lm/mha_sinusoidal_chunk_4.pt"

HF_REPO = "Exploration-Lab/CS-781-Capstone"

TRAIN_JSON = "Phase_7_PCR/Train/train.json"
TRAIN_CANDIDATES_JSON = "Phase_7_PCR/Train/train_candidates.json"
DEV_JSON = "Phase_7_PCR/Dev/dev.json"
DEV_CANDIDATES_JSON = "Phase_7_PCR/Dev/dev_candidates.json"
TEST_JSON = "Phase_7_PCR/Test/test.json"
TEST_CANDIDATES_JSON = "Phase_7_PCR/Test/test_candidates.json"

CHECKPOINT_DIR = "/kaggle/working/PCR_LinearAttention_LoRA_KD_Random_Negatives"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

RANDOM_NEGATIVE_CACHE_PATH = os.path.join(
    CHECKPOINT_DIR, "random_train_negative_map.json"
)
HISTORY_PATH = os.path.join(
    CHECKPOINT_DIR, "history.json"
)
BEST_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR, "best_student.pt"
)
TEST_PREDICTIONS_PATH = os.path.join(
    CHECKPOINT_DIR, "test_retrieval_predictions.json"
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("device:", device)
print("MAX_CHUNKS:", MAX_CHUNKS)
print("CHUNK_BATCH_SIZE:", CHUNK_BATCH_SIZE)
print("NUM_NEGATIVES:", NUM_NEGATIVES)
print("BATCH_SIZE:", BATCH_SIZE)
print("TEACHER_N_LAYERS:", TEACHER_N_LAYERS)
print("STUDENT_N_LAYERS:", STUDENT_N_LAYERS)
print("KD_WEIGHT:", KD_WEIGHT)
print("TEACHER_CHECKPOINT_PATH:", TEACHER_CHECKPOINT_PATH)
print("USE_GRAD_CHECKPOINTING:", USE_GRAD_CHECKPOINTING)
print("USE_AMP:", USE_AMP)


device: cuda
MAX_CHUNKS: 8
CHUNK_BATCH_SIZE: 8
NUM_NEGATIVES: 4
BATCH_SIZE: 8
TEACHER_N_LAYERS: 12
STUDENT_N_LAYERS: 6
KD_WEIGHT: 0.5
TEACHER_CHECKPOINT_PATH: /kaggle/input/datasets/rogue2222/kerenel-p7/PCR_LinearAttention_LoRA_Random_Negatives/best.pt
USE_GRAD_CHECKPOINTING: True
USE_AMP: True


In [3]:
# -------------------------
# Utilities
# -------------------------
def clear_cuda_memory():
    gc.collect()
    if torch.cuda.is_available():
        for gpu_id in range(torch.cuda.device_count()):
            with torch.cuda.device(gpu_id):
                torch.cuda.empty_cache()
                torch.cuda.ipc_collect()
                torch.cuda.reset_peak_memory_stats()

def gpu_memory_report(prefix=""):
    if not torch.cuda.is_available():
        print(prefix + "CUDA unavailable")
        return
    for i in range(torch.cuda.device_count()):
        with torch.cuda.device(i):
            allocated = torch.cuda.memory_allocated() / 1024**3
            reserved = torch.cuda.memory_reserved() / 1024**3
            free, total = torch.cuda.mem_get_info()
            print(
                f"{prefix}GPU {i}: allocated={allocated:.2f} GB, "
                f"reserved={reserved:.2f} GB, "
                f"free={free/1024**3:.2f} GB / total={total/1024**3:.2f} GB"
            )

clear_cuda_memory()
gpu_memory_report("Initial | ")


Initial | GPU 0: allocated=0.00 GB, reserved=0.00 GB, free=14.46 GB / total=14.56 GB
Initial | GPU 1: allocated=0.00 GB, reserved=0.00 GB, free=14.46 GB / total=14.56 GB


In [4]:
# ------------------------------------------------------------
# Hugging Face login
# ------------------------------------------------------------

from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Hugging Face login successful.")
# ------------------------------------------------------------
# Download Phase-7 PCR files
# ------------------------------------------------------------

TRAIN_FILES = [
    "Phase_7_PCR/Train/train.json",
    "Phase_7_PCR/Train/train_candidates.json",
]
DEV_FILES = [
    "Phase_7_PCR/Dev/dev.json",
    "Phase_7_PCR/Dev/dev_candidates.json",
]
TEST_FILES = [
    "Phase_7_PCR/Test/test.json",
    "Phase_7_PCR/Test/test_candidates.json",
]

def download_files(files):
    paths = {}
    for filename in files:
        paths[filename] = hf_hub_download(
            repo_id=HF_DATASET_REPO,
            filename=filename,
            repo_type="dataset",
            token=HF_TOKEN,
        )
        print(filename, "->", paths[filename])
    return paths

train_file_paths = download_files(TRAIN_FILES)
dev_file_paths = download_files(DEV_FILES)
test_file_paths = download_files(TEST_FILES)
# ------------------------------------------------------------
# Load raw JSON
# ------------------------------------------------------------

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_data_raw = load_json(train_file_paths["Phase_7_PCR/Train/train.json"])
train_candidates_raw = load_json(train_file_paths["Phase_7_PCR/Train/train_candidates.json"])

dev_data_raw = load_json(dev_file_paths["Phase_7_PCR/Dev/dev.json"])
dev_candidates_raw = load_json(dev_file_paths["Phase_7_PCR/Dev/dev_candidates.json"])

test_data_raw = load_json(test_file_paths["Phase_7_PCR/Test/test.json"])
test_candidates_raw = load_json(test_file_paths["Phase_7_PCR/Test/test_candidates.json"])

print("Train queries:", len(train_data_raw))
print("Train candidates:", len(train_candidates_raw))
print("Dev queries:", len(dev_data_raw))
print("Dev candidates:", len(dev_candidates_raw))
print("Test queries:", len(test_data_raw))
print("Test candidates:", len(test_candidates_raw))


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


Phase_7_PCR/Train/train.json: reconstructing file:   0%|          |  0.00B / 33.2MB            

Phase_7_PCR/Train/train.json: downloading bytes:           |  0.00B            

Phase_7_PCR/Train/train.json -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_7_PCR/Train/train.json


Phase_7_PCR/Train/train_candidates.json: reconstructing file:   0%|          |  0.00B /  160MB            

... (8 outputs left out)

Phase_7_PCR/Test/test.json -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_7_PCR/Test/test.json


Phase_7_PCR/Test/test_candidates.json: reconstructing file:   0%|          |  0.00B / 71.1MB            

Phase_7_PCR/Test/test_candidates.json: downloading bytes:           |  0.00B            

Phase_7_PCR/Test/test_candidates.json -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_7_PCR/Test/test_candidates.json
Train queries: 827
Train candidates: 4320
Dev queries: 118
Dev candidates: 1023
Test queries: 237
Test candidates: 1727


In [5]:
class Node:
    def __init__(self,value, previous, next, index):
        self.value=value
        self.previous=previous
        self.next=next
        self.index=index


class LinkedArray:
    def __init__(self, items):
        self._array=[None for _ in range(len(items))]
        previous=None
        for i in range(len(items)):
            node=Node(items[i], previous, None, i)
            if previous is not None:
                node.previous=previous
                previous.next=node
            self._array[i]=node
            previous=node

    def get_previous_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].previous == None:
            return None
        return self._array[index].previous.index

    def get_next_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].next == None:
            return None
        return self._array[index].next.index

    def get_by_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index] == None:
            return None
        return self._array[index].value

    def get_second_next_index(self, index):
        if self._array[index] == None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].next == None or self._array[index].next.next == None:
            return None
        return self._array[index].next.next.index

    def len(self):
        return len(self._array)
    def exists(self, index):
        return (
            0 <= index < len(self._array)
            and self._array[index] is not None
        )

    def replace_pair(self, index, new_item):
        if index > len(self._array)-2 or self._array[index] == None or self._array[index].next == None:
            raise ValueError("Invalid index ",index,new_item)
        self._array[index].value = new_item
        self._array[self._array[index].next.index] = None
        self._array[index].next = self._array[index].next.next
        if self._array[index].next is not None:
            self._array[index].next.previous = self._array[index]


class MaxPriorityMap:
    def __init__(self, heap_key, map_key):
        self._heap_key = heap_key
        self._map_key = map_key
        self._heap = []
        # Map of keys to indexes in the heap
        self._map = {}

    def get_max(self):
        if len(self._heap) == 0:
            raise ValueError("the heap is empty")
        return self._heap[0]

    def push(self, item):
        self._heap.append(item)
        map_key = self._map_key(item)
        if map_key in self._map:
            raise ValueError("Map key already exist")
        self._map[self._map_key(item)] = len(self._heap) - 1
        self._heapify_up(len(self._heap) - 1)

    def pop(self):
        if len(self._heap) == 0:
            raise ValueError("the heap is empty")
        self._swap(0, len(self._heap) - 1)
        item = self._heap.pop()
        map_key = self._map_key(item)
        self._map.pop(map_key)
        self._heapify_down(0)
        return item

    def contains(self, map_key):
        return map_key in self._map

    def len(self):
        if len(self._heap) != len(self._map):
            raise ValueError('Invalid state of the priority map')
        return len(self._heap)

    def delete_by_map_key(self, map_key):
        if map_key not in self._map:
            raise ValueError("Map key does not exist")
        heap_index = self._map[map_key]
        item = self._heap[heap_index]
        self._swap(heap_index, len(self._heap) - 1)
        self._heap.pop()
        self._map.pop(map_key)
        if (heap_index < len(self._heap)):
            self._heapify_down(heap_index)
            self._heapify_up(heap_index)
        return item

    def _heapify_up(self, i):
        while i > 0 and self._heap_key(self._heap[i]) > self._heap_key(self._heap[self._parent(i)]):
            self._swap(i, self._parent(i))
            i = self._parent(i)

    def _heapify_down(self, i):
        largest = i
        left = self._left_child(i)
        right = self._right_child(i)

        if left < len(self._heap) and self._heap_key(self._heap[left]) > self._heap_key(self._heap[largest]):
            largest = left

        if right < len(self._heap) and self._heap_key(self._heap[right]) > self._heap_key(self._heap[largest]):
            largest = right

        if largest != i:
            self._swap(i, largest)
            self._heapify_down(largest)

    def _parent(self, i):
        return (i - 1) // 2

    def _left_child(self, i):
        return 2 * i + 1

    def _right_child(self, i):
        return 2 * i + 2

    def _swap(self, i, j):
        self._map[self._map_key(self._heap[i])] = j
        self._map[self._map_key(self._heap[j])] = i
        self._heap[i], self._heap[j] = self._heap[j], self._heap[i]


class StatsEntry:
    def __init__(self, pair, positions):
        self.pair = pair
        self.positions = positions

class TokenizerTrainer:
        def __init__(self, tokens_list, itos, num_merges):
            self.tokens_list = tokens_list
            self.itos = itos
            self._num_merges = num_merges
            self.stoi={}

        def train(self, next_token):
            self._positions=[LinkedArray(tokens) for tokens in self.tokens_list]
            self.get_stats()
            merges_done=0
            print("requested merges",self._num_merges)
            while  merges_done<self._num_merges:
                if self._stats.len() == 0:
                   print("STOPPED: stats became empty")
                   print("Merges completed:", merges_done)
                   break
                merge_stat = self._stats.pop()
                (current_token , next_token) = self.update_itos_and_stoi(next_token, merge_stat)
                #iterate through posistions replace with merge pair
                for position in list(merge_stat.positions):

                    if position not in merge_stat.positions:
                        continue
                    input_index = position[0]
                    token_index = position[1]
                    # print("current_token",current_token)
                    if not self._positions[input_index].exists(token_index):
                        merge_stat.positions.discard(position)
                        continue
                    self.update_right_token(input_index, token_index, merge_stat, current_token)
                    self.update_left_token(input_index, token_index, merge_stat, current_token)
                    self._positions[input_index].replace_pair(token_index, current_token)
                    merge_stat.positions.remove((input_index, token_index))
                merges_done += 1
            return self.itos


        def update_right_token(self, input_index, token_index, merge_stat, current_token):
           position = self._positions[input_index]
           next_token_idx = position.get_next_index(token_index)
           right_token_idx = position.get_second_next_index(token_index)
           if right_token_idx == None:
               return
           pair = (merge_stat.pair[1], position.get_by_index(right_token_idx))
           self.remove_pair_from_stats(merge_stat, pair, input_index, next_token_idx)
           new_pair = (current_token, pair[1])
           self.add_position_to_pair(new_pair, input_index, token_index)

        def update_left_token(self, input_index, token_index, merge_stat, current_token):
           position = self._positions[input_index]
           left_token_idx = position.get_previous_index(token_index)
           if left_token_idx == None:
               return
           pair = ( position.get_by_index(left_token_idx), merge_stat.pair[0])
           self.remove_pair_from_stats(merge_stat, pair, input_index, left_token_idx)
           new_pair = (pair[0] , current_token)
           self.add_position_to_pair(new_pair, input_index, left_token_idx)

        def add_position_to_pair(self, new_pair, input_index, token_index):
            if self._stats.contains(new_pair):
                stat = self._stats.delete_by_map_key(new_pair)
            else:
                stat = StatsEntry(new_pair, set())

            stat.positions.add((input_index, token_index))
            self._stats.push(stat)


        def remove_pair_from_stats(self, merge_stat, pair, input_index, token_index):
           if pair == merge_stat.pair:
               merge_stat.positions.discard((input_index, token_index))
           else:
                if self._stats.contains(pair):
                    stat = self._stats.delete_by_map_key(pair)
                    stat.positions.discard((input_index, token_index))
                    if len(stat.positions) != 0:
                        self._stats.push(stat)


        def update_itos_and_stoi(self, next_token, merge_stat):
           merge_pair = self.itos[merge_stat.pair[0]] + self.itos[merge_stat.pair[1]]
           if merge_pair in self.stoi:
               current_token = self.stoi[merge_pair]
           else:
               current_token = next_token
               self.itos[current_token] = merge_pair
               self.stoi[merge_pair] = current_token
               next_token = current_token+1
           return  (current_token , next_token)

        def get_stats(self):
             self._stats = MaxPriorityMap(
                                heap_key=lambda item: (len(item.positions),tuple(-x for x in item.pair)),
                                map_key=lambda item: item.pair
                            )
             stats={}
             for  text_id in range(len(self.tokens_list)):
                 tokens = self.tokens_list[text_id]
                 for idx in range(len(tokens)-1):
                     pair = (tokens[idx], tokens[idx+1])
                     if pair not in stats:
                         stats[pair]=StatsEntry(pair, set())
                     stats[pair].positions.add((text_id, idx))
             for key in stats.keys():
                self._stats.push(stats[key])

class TokenNode:
    def __init__(self, byte_value, token, parent, children):
        self.parent = parent
        self.byte_value = byte_value
        self.token = token
        self.children = children

class ToTokensConverter:

    def __init__(self, itos):
        self.itos = itos
        self.unk_token=1
        self._init_token_tree()

    def to_tokens(self, strings):
        return [
            self._to_tokens(string)
            for string in strings
        ]

    def _to_tokens(self, string):

        words = string.split()
        tokens = []
        for word in words:
            data = word.encode("utf-8") + b"</w>"
            word_tokens = self._tokenize_word(data)
            tokens.extend(word_tokens)
        return tokens

    def _tokenize_word(self, data):

        tokens = []

        byte_index = 0

        while byte_index < len(data):

            current_nodes = self._token_roots

            last_node_with_token = None
            last_index_with_token = byte_index

            while byte_index < len(data):

                byte_value = data[byte_index]

                if byte_value in current_nodes:

                    node = current_nodes[byte_value]

                    if node.token is not None:
                        last_node_with_token = node
                        last_index_with_token = byte_index

                    current_nodes = node.children
                    byte_index += 1

                else:
                    break

            if last_node_with_token is None:
                tokens.append(self.unk_token)

                # Move forward by one byte
                byte_index += 1

            else:
                tokens.append(last_node_with_token.token)
                byte_index = last_index_with_token + 1

        return tokens

    def _init_token_tree(self):

        self._token_roots = {}

        for token, token_bytes in self.itos.items():

            if token < 4:
                continue

            current_nodes = self._token_roots
            last_node = None

            for byte_value in token_bytes:

                if byte_value in current_nodes:

                    last_node = current_nodes[byte_value]

                else:

                    last_node = TokenNode(
                        byte_value,
                        None,
                        last_node,
                        {}
                    )

                    current_nodes[byte_value] = last_node

                current_nodes = last_node.children

            if last_node.token is not None:
                raise KeyError("The token is already taken")

            last_node.token = token

class BPETokenizer:
    def __init__(self, num_merges):
        self._num_merges = num_merges
        self.itos = {}
    def get_vocab_size(self):
        return len(self.itos)
    def train(self, texts):
        self.create_base_vocab()

        tokens_list = self.get_ids_from_text(texts)
        # for tokens in tokens_list:
        #     print(tokens)
        trainer = TokenizerTrainer(
                        tokens_list,
                        self.itos,
                        self._num_merges
                    )

        trainer.train(261)

        return self.itos, tokens_list

    def encode(self, text):
        converter = ToTokensConverter(self.itos)
        return converter.to_tokens([text])[0]

    def decode(self, tokens):
        return (
                b"".join(
                    self.itos[token]
                    for token in tokens
                )
                .decode("utf-8", errors="replace")
                .replace("</w>", " ")
                .strip()
            )

    def create_base_vocab(self):
        self.itos = {
            0: b"<pad>",
            1: b"<unk>",
            2: b"<s>",
            3: b"</s>",
            260 : b"</w>"
        }
        for i in range(256):
            self.itos[i + 4] = bytes([i])

    def get_ids_from_text(self, texts):
        tokens_list=[]
        for text in texts:
            words = text.split()
            for word in words:
                word_bytes = [byte + 4 for byte in word.encode("utf-8")]
                word_bytes.append(260)
                tokens_list.append(word_bytes)
        return tokens_list


In [6]:
# ------------------------------------------------------------
# Load the existing BPE tokenizer
# ------------------------------------------------------------
import pickle
BPE_PATH = "/kaggle/input/datasets/rogue2222/tokenizer-results/project_bpe_16k.pkl"

with open(BPE_PATH, "rb") as f:
    tokenizer = pickle.load(f)



converter = ToTokensConverter(tokenizer.itos)

PAD_ID = 0
UNK_ID = 1
BOS_ID = 2
EOS_ID = 3
VOCAB_SIZE = 8000

print("Special IDs:", PAD_ID, UNK_ID, BOS_ID, EOS_ID)


Special IDs: 0 1 2 3


In [7]:

def convert_text_to_ids(text):
    token_ids = converter.to_tokens([text])[0]
    if len(token_ids) == 0:
        return [UNK_ID]
    token_ids = [int(x) for x in token_ids]
    bad_ids = [x for x in token_ids if x < 0 or x >= VOCAB_SIZE]
    if bad_ids:
        raise ValueError(f"Invalid token ids: {bad_ids[:10]}")
    return token_ids

print("Vocabulary size:", VOCAB_SIZE)


Vocabulary size: 8000


In [8]:
# ------------------------------------------------------------
# Data processing — same style as the reference notebook
# No normalization is performed.
# ------------------------------------------------------------

def case_text_to_string(text):
    if isinstance(text, str):
        return text

    if isinstance(text, list):
        parts = []
        for item in text:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict) and "text" in item:
                parts.append(str(item["text"]))
            else:
                raise TypeError(
                    f"Unsupported text item: {type(item)}"
                )
        return " ".join(parts)

    raise TypeError(f"Unsupported text type: {type(text)}")


def prepare_records(raw_records, labeled=False):
    records = {}

    if isinstance(raw_records, dict):
        iterator = raw_records.items()
    elif isinstance(raw_records, list):
        iterator = (
            (
                row.get(
                    "id",
                    row.get("query_id", row.get("case_id")),
                ),
                row,
            )
            for row in raw_records
        )
    else:
        raise TypeError("Dataset must be a dict or list.")

    for raw_id, row in iterator:
        if raw_id is None:
            raise ValueError("A record has no case ID.")

        case_id = str(raw_id)

        if case_id in records:
            raise ValueError(f"Duplicate case ID: {case_id}")

        text = case_text_to_string(row["text"])

        if not text.strip():
            raise ValueError(f"Empty text for case {case_id}")

        item = {"text": text}

        if labeled:
            positives = row["relevant_candidates"]
            if not isinstance(positives, list):
                raise ValueError(
                    "relevant_candidates must be a list."
                )
            item["relevant_candidates"] = [
                str(cid) for cid in positives
            ]

        records[case_id] = item

    return records


# IMPORTANT:
# train_data/dev_data are QUERIES.
# train_candidates/dev_candidates are CANDIDATES.

train_data = prepare_records(
    train_data_raw,
    labeled=True,
)
train_candidates = prepare_records(
    train_candidates_raw,
)

dev_data = prepare_records(
    dev_data_raw,
    labeled=True,
)
dev_candidates = prepare_records(
    dev_candidates_raw,
)

test_data = prepare_records(test_data_raw)
test_candidates = prepare_records(test_candidates_raw)

print("Train:", len(train_data), "queries,", len(train_candidates), "candidates")
print("Dev:  ", len(dev_data), "queries,", len(dev_candidates), "candidates")
print("Test: ", len(test_data), "queries,", len(test_candidates), "candidates")


Train: 827 queries, 4320 candidates
Dev:   118 queries, 1023 candidates
Test:  237 queries, 1727 candidates


In [9]:
# -------------------------
# Chunking
# -------------------------
def chunk_token_ids(token_ids):
    content = list(token_ids)

    if not content:
        content = [UNK_ID]

    starts = list(range(
        0,
        max(len(content) - CHUNK_CONTENT_LEN, 0) + 1,
        CHUNK_STRIDE,
    ))

    last_start = max(len(content) - CHUNK_CONTENT_LEN, 0)
    if not starts or starts[-1] != last_start:
        starts.append(last_start)

    starts = sorted(set(starts))

    if len(starts) > MAX_CHUNKS:
        # Preserve first and last chunks while selecting intermediate chunks
        # approximately uniformly.
        idx = np.linspace(0, len(starts) - 1, MAX_CHUNKS).round().astype(int)
        starts = [starts[i] for i in idx]
        starts = sorted(set(starts))

    chunks = []
    for start in starts:
        piece = content[start:start + CHUNK_CONTENT_LEN]
        chunk = [BOS_ID] + piece + [EOS_ID]
        chunk = chunk[:MAX_SEQ_LEN]

        if len(chunk) < MAX_SEQ_LEN:
            chunk = chunk + [PAD_ID] * (MAX_SEQ_LEN - len(chunk))

        chunks.append(chunk)

    while len(chunks) < MAX_CHUNKS:
        chunks.append([PAD_ID] * MAX_SEQ_LEN)

    return torch.tensor(chunks[:MAX_CHUNKS], dtype=torch.long)

def encode_record_text(text):
    ids = convert_text_to_ids(text)
    return chunk_token_ids(ids)

# Precompute chunks. No text normalization is performed.
def build_chunk_records(records, desc):
    result = {}
    for doc_id, row in tqdm(records.items(), desc=desc):
        result[doc_id] = {
            "chunks": encode_record_text(row["text"])
        }
    return result

train_query_records = build_chunk_records(train_data, "Chunking train queries")
train_candidate_records = build_chunk_records(
    train_candidates, "Chunking train candidates"
)
dev_query_records = build_chunk_records(dev_data, "Chunking dev queries")
dev_candidate_records = build_chunk_records(
    dev_candidates, "Chunking dev candidates"
)
test_query_records = build_chunk_records(test_data, "Chunking test queries")
test_candidate_records = build_chunk_records(
    test_candidates, "Chunking test candidates"
)

print("Example query chunks:", next(iter(train_query_records.values()))["chunks"].shape)


Chunking train queries:   0%|          | 0/827 [00:00<?, ?it/s]

Chunking train candidates:   0%|          | 0/4320 [00:00<?, ?it/s]

Chunking dev queries:   0%|          | 0/118 [00:00<?, ?it/s]

Chunking dev candidates:   0%|          | 0/1023 [00:00<?, ?it/s]

Chunking test queries:   0%|          | 0/237 [00:00<?, ?it/s]

Chunking test candidates:   0%|          | 0/1727 [00:00<?, ?it/s]

Example query chunks: torch.Size([8, 1024])


In [10]:
# -------------------------
# Random negatives
# -------------------------
def build_random_negative_map(query_data, candidate_data, seed=SEED):
    rng = random.Random(seed)
    query_ids = list(query_data.keys())
    candidate_ids = list(candidate_data.keys())

    negative_map = {}

    for qid in tqdm(query_ids, desc="Sampling random negatives"):
        positives = list(dict.fromkeys(
            query_data[qid].get("relevant_candidates", [])
        ))
        positive_set = set(positives)

        eligible = [
            cid for cid in candidate_ids
            if cid not in positive_set
        ]

        if len(eligible) < NUM_NEGATIVES:
            raise RuntimeError(
                f"Query {qid} has only {len(eligible)} eligible negatives."
            )

        negative_map[qid] = rng.sample(eligible, NUM_NEGATIVES)

    return negative_map

if os.path.exists(RANDOM_NEGATIVE_CACHE_PATH):
    with open(RANDOM_NEGATIVE_CACHE_PATH, "r", encoding="utf-8") as f:
        random_negative_map = json.load(f)
    print("Loaded cached random negatives.")
else:
    random_negative_map = build_random_negative_map(
        train_data,
        train_candidates,
        seed=SEED,
    )
    with open(RANDOM_NEGATIVE_CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump(random_negative_map, f, indent=2)
    print("Saved random negatives.")

print("Example:", next(iter(random_negative_map.items())))


Sampling random negatives:   0%|          | 0/827 [00:00<?, ?it/s]

Saved random negatives.
Example: ('0000000599', ['0000432540', '0000088171', '0001069919', '0000952560'])


In [11]:
# -------------------------
# PCR Dataset
# -------------------------
class PCRDataset(Dataset):
    def __init__(
        self,
        query_data,
        query_records,
        candidate_records,
        negative_map,
        num_negatives=4,
    ):
        self.query_data = query_data
        self.query_records = query_records
        self.candidate_records = candidate_records
        self.negative_map = negative_map
        self.num_negatives = num_negatives

        self.query_ids = [
            qid for qid, row in query_data.items()
            if row.get("relevant_candidates")
        ]

    def __len__(self):
        return len(self.query_ids)

    def __getitem__(self, idx):
        qid = self.query_ids[idx]
        row = self.query_data[qid]

        positives = list(dict.fromkeys(row["relevant_candidates"]))
        negatives = self.negative_map[qid][:self.num_negatives]

        candidate_ids = positives + negatives
        positive_mask = [True] * len(positives) + [False] * len(negatives)

        query_chunks = self.query_records[qid]["chunks"]

        candidate_chunks = torch.stack([
            self.candidate_records[cid]["chunks"]
            for cid in candidate_ids
        ])

        return {
            "qid": qid,
            "query_chunks": query_chunks,
            "candidate_chunks": candidate_chunks,
            "candidate_ids": candidate_ids,
            "positive_mask": torch.tensor(
                positive_mask, dtype=torch.bool
            ),
        }

train_dataset = PCRDataset(
    train_data,
    train_query_records,
    train_candidate_records,
    random_negative_map,
    NUM_NEGATIVES,
)

def collate_fn(batch):

    return batch

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
)

print("Training queries:", len(train_dataset))


Training queries: 817


In [12]:
# ============================================================
# Supplied GQA Linear Attention kernel
# ============================================================

def chunked_gqa_linear_attn(
    q,
    k,
    v,
    chunk=128,
    eps=1e-6,
    padding_mask=None,
):
    """
    q: (B, Hq, T, F)
    k: (B, Hkv, T, F)
    v: (B, Hkv, T, D)

    All attention-kernel calculations are performed in float32.

    IMPORTANT:
        padding_mask=True means PAD.
        This matches the supplied kernel implementation.
    """
    B, Hq, T, Fd = q.shape

    Hkv = k.shape[1]
    D = v.shape[-1]

    R = Hq // Hkv

    if padding_mask is not None:
        valid = (
            ~padding_mask.bool()
        )[:, None, :, None]

        k = k * valid.to(k.dtype)
        v = v * valid.to(v.dtype)

    q = q.view(
        B,
        Hkv,
        R,
        T,
        Fd,
    )

    S = q.new_zeros(
        B,
        Hkv,
        Fd,
        D,
    )

    z = q.new_zeros(
        B,
        Hkv,
        Fd,
    )

    tri = torch.tril(
        torch.ones(
            chunk,
            chunk,
            dtype=torch.bool,
            device=q.device,
        )
    )

    outs = []

    for s in range(
        0,
        T,
        chunk,
    ):
        qc = q[
            :,
            :,
            :,
            s:s + chunk,
        ]

        kc = k[
            :,
            :,
            s:s + chunk,
        ]

        vc = v[
            :,
            :,
            s:s + chunk,
        ]

        c = qc.shape[3]

        a = (
            qc
            @ kc.unsqueeze(2).transpose(
                -1,
                -2,
            )
        ).masked_fill(
            ~tri[:c, :c],
            0.0,
        )

        num = (
            qc
            @ S.unsqueeze(2)
            + a @ vc.unsqueeze(2)
        )

        den = (
            (
                qc
                * z[
                    :,
                    :,
                    None,
                    None,
                    :,
                ]
            ).sum(
                -1,
                keepdim=True,
            )
            + a.sum(
                -1,
                keepdim=True,
            )
        )

        outs.append(
            num / (den + eps)
        )

        S = (
            S
            + kc.transpose(
                -1,
                -2,
            ) @ vc
        )

        z = (
            z
            + kc.sum(dim=2)
        )

    return (
        torch.cat(
            outs,
            dim=3,
        ).reshape(
            B,
            Hq,
            T,
            D,
        ),
        (S, z),
    )


class LinearAttention(nn.Module):
    def __init__(
        self,
        d_model,
        n_heads,
        n_kv_heads,
        dropout=0.0,
        feat_dim=32,
        chunk=128,
    ):
        super().__init__()

        assert (
            d_model % n_heads == 0
            and n_heads % n_kv_heads == 0
        )

        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads

        self.d_head = (
            d_model // n_heads
        )

        self.chunk = chunk
        self.feat_dim = feat_dim

        self.q_proj = nn.Linear(
            d_model,
            n_heads * self.d_head,
            bias=False,
        )

        self.k_proj = nn.Linear(
            d_model,
            n_kv_heads * self.d_head,
            bias=False,
        )

        self.v_proj = nn.Linear(
            d_model,
            n_kv_heads * self.d_head,
            bias=False,
        )

        self.out_proj = nn.Linear(
            n_heads * self.d_head,
            d_model,
            bias=False,
        )

        # phi(x) = (LayerNorm(gamma*x + beta))^2
        self.q_kernel = nn.Linear(
            self.d_head,
            feat_dim,
            bias=True,
        )

        self.k_kernel = nn.Linear(
            self.d_head,
            feat_dim,
            bias=True,
        )

        self.q_norm = nn.LayerNorm(
            feat_dim
        )

        self.k_norm = nn.LayerNorm(
            feat_dim
        )

        self.resid_dropout = nn.Dropout(
            dropout
        )

    def _project(self, x):
        B, T, _ = x.shape

        q = self.q_proj(x).view(
            B,
            T,
            self.n_heads,
            self.d_head,
        ).transpose(1, 2)

        k = self.k_proj(x).view(
            B,
            T,
            self.n_kv_heads,
            self.d_head,
        ).transpose(1, 2)

        v = self.v_proj(x).view(
            B,
            T,
            self.n_kv_heads,
            self.d_head,
        ).transpose(1, 2)

        q_phi = (
            self.q_norm(
                self.q_kernel(q)
            ).float()
            ** 2
        )

        k_phi = (
            self.k_norm(
                self.k_kernel(k)
            ).float()
            ** 2
        )

        return (
            q_phi,
            k_phi,
            v.float(),
        )

    def forward(
        self,
        x,
        padding_mask=None,
        return_state=False,
    ):
        B, T, C = x.shape

        q_phi, k_phi, v = (
            self._project(x)
        )

        # The supplied kernel is explicitly float32.
        with torch.autocast(
            device_type="cuda",
            enabled=False,
        ):
            out, state = (
                chunked_gqa_linear_attn(
                    q_phi,
                    k_phi,
                    v,
                    self.chunk,
                    padding_mask=padding_mask,
                )
            )

        out = F.rms_norm(
            out.to(x.dtype),
            (self.d_head,),
        )

        out = self.out_proj(
            out.transpose(
                1,
                2,
            ).reshape(
                B,
                T,
                C,
            )
        )

        out = self.resid_dropout(
            out
        )

        return (
            (out, state)
            if return_state
            else out
        )

    @torch.no_grad()
    def step(
        self,
        x_t,
        S,
        z,
        eps=1e-6,
    ):
        """
        One-token decoding.

        x_t: (B,1,C)
        S:   (B,Hkv,F,D), float32
        z:   (B,Hkv,F), float32
        """
        B = x_t.shape[0]

        q_phi, k_phi, v = (
            self._project(x_t)
        )

        S = (
            S
            + k_phi.transpose(
                -1,
                -2,
            ) @ v
        )

        z = (
            z
            + k_phi.squeeze(2)
        )

        qg = q_phi.reshape(
            B,
            self.n_kv_heads,
            self.n_rep,
            self.feat_dim,
        )

        numerator = torch.einsum(
            "bhrf,bhfd->bhrd",
            qg,
            S,
        )

        denominator = (
            qg
            * z.unsqueeze(2)
        ).sum(
            -1,
            keepdim=True,
        )

        out = numerator / (
            denominator + eps
        )

        out = F.rms_norm(
            out.to(x_t.dtype),
            (self.d_head,),
        )

        out = out.reshape(
            B,
            1,
            self.n_heads
            * self.d_head,
        )

        out = self.out_proj(
            out
        )

        return (
            out,
            S,
            z,
        )


In [13]:
# -------------------------
# Positional encoding
# -------------------------
# LinearAttention itself does not apply RoPE.
# PCR uses the supplied decoder's sinusoidal positional encoding.


In [14]:
# ============================================================
# Decoder-only Transformer LM with LinearAttention
# ============================================================

@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 512
    n_layers: int = 12
    n_heads: int = 8
    n_kv_heads: int = 8
    d_ff: int = 4096
    max_seq_len: int = 1024
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"
    feat_dim: int = 32
    chunk: int = 128


def sinusoidal_positional_encoding(
    max_len,
    d_model,
):
    pos = torch.arange(
        max_len,
        dtype=torch.float32,
    )[:, None]

    i = torch.arange(
        d_model // 2,
        dtype=torch.float32,
    )[None, :]

    ang = pos / (
        10000
        ** (
            2 * i / d_model
        )
    )

    pe = torch.zeros(
        max_len,
        d_model,
    )

    pe[:, 0::2] = torch.sin(
        ang
    )
    pe[:, 1::2] = torch.cos(
        ang
    )

    return pe


class FeedForward(nn.Module):
    def __init__(
        self,
        d_model,
        d_ff,
        dropout,
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(
                d_model,
                d_ff,
            ),
            nn.GELU(),
            nn.Linear(
                d_ff,
                d_model,
            ),
            nn.Dropout(
                dropout
            ),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(
        self,
        cfg,
    ):
        super().__init__()

        self.ln1 = nn.LayerNorm(
            cfg.d_model
        )

        self.attn = LinearAttention(
            cfg.d_model,
            cfg.n_heads,
            cfg.n_kv_heads,
            cfg.dropout,
            cfg.feat_dim,
            cfg.chunk,
        )

        self.ln2 = nn.LayerNorm(
            cfg.d_model
        )

        self.ffn = FeedForward(
            cfg.d_model,
            cfg.d_ff,
            cfg.dropout,
        )

    def forward(
        self,
        x,
        padding_mask=None,
        return_state=False,
    ):
        if return_state:
            a, state = self.attn(
                self.ln1(x),
                padding_mask=padding_mask,
                return_state=True,
            )
        else:
            a = self.attn(
                self.ln1(x),
                padding_mask=padding_mask,
            )
            state = None

        x = x + a

        x = x + self.ffn(
            self.ln2(x)
        )

        return (
            (x, state)
            if return_state
            else x
        )

    @torch.no_grad()
    def step(
        self,
        x_t,
        state,
    ):
        S, z = state

        a, S, z = (
            self.attn.step(
                self.ln1(x_t),
                S,
                z,
            )
        )

        x = x_t + a

        x = x + self.ffn(
            self.ln2(x)
        )

        return x, (S, z)


class DecoderLM(nn.Module):
    def __init__(
        self,
        cfg,
    ):
        super().__init__()

        self.cfg = cfg

        self.tok_emb = nn.Embedding(
            cfg.vocab_size,
            cfg.d_model,
        )

        if (
            cfg.pos_emb_type
            == "sinusoidal"
        ):
            self.register_buffer(
                "pos_emb",
                sinusoidal_positional_encoding(
                    cfg.max_seq_len,
                    cfg.d_model,
                ),
                persistent=False,
            )

        self.blocks = nn.ModuleList(
            [
                Block(cfg)
                for _ in range(
                    cfg.n_layers
                )
            ]
        )

        self.ln_f = nn.LayerNorm(
            cfg.d_model
        )

        self.head = nn.Linear(
            cfg.d_model,
            cfg.vocab_size,
            bias=False,
        )

        self.head.weight = (
            self.tok_emb.weight
        )

        with torch.no_grad():
            nn.init.uniform_(
                self.tok_emb.weight,
                -1
                / math.sqrt(
                    cfg.d_model
                ),
                1
                / math.sqrt(
                    cfg.d_model
                ),
            )

    def encode(
        self,
        idx,
        padding_mask=None,
    ):
        x = self.tok_emb(idx)

        if (
            self.cfg.pos_emb_type
            == "sinusoidal"
        ):
            T = idx.shape[1]

            x = x + self.pos_emb[
                :T
            ].unsqueeze(0)

        for block in self.blocks:
            if (
                self.training
                and USE_GRAD_CHECKPOINTING
                and torch.is_grad_enabled()
            ):
                x = torch.utils.checkpoint.checkpoint(
                    block,
                    x,
                    padding_mask,
                    use_reentrant=False,
                )
            else:
                x = block(
                    x,
                    padding_mask=padding_mask,
                )

        return self.ln_f(x)

    def forward(
        self,
        idx,
        padding_mask=None,
    ):
        x = self.encode(
            idx,
            padding_mask=padding_mask,
        )

        return self.head(x)

    @torch.no_grad()
    def prefill(self, idx):
        x = self.tok_emb(idx)

        if (
            self.cfg.pos_emb_type
            == "sinusoidal"
        ):
            T = idx.shape[1]
            x = x + self.pos_emb[
                :T
            ].unsqueeze(0)

        states = []

        for block in self.blocks:
            x, state = block(
                x,
                return_state=True,
            )
            states.append(state)

        logits = self.head(
            self.ln_f(x[:, -1:])
        )

        return logits, states

    @torch.no_grad()
    def step(
        self,
        tok,
        pos,
        states,
    ):
        x = self.tok_emb(tok)

        if (
            self.cfg.pos_emb_type
            == "sinusoidal"
        ):
            x = x + self.pos_emb[
                pos:pos + 1
            ].unsqueeze(0)

        new_states = []

        for block, state in zip(
            self.blocks,
            states,
        ):
            x, new_state = (
                block.step(
                    x,
                    state,
                )
            )
            new_states.append(
                new_state
            )

        logits = self.head(
            self.ln_f(x)
        )

        return logits, new_states


In [15]:
# -------------------------
# Standard additive LoRA
# -------------------------
class LoRALinear(nn.Module):
    """
    Standard additive LoRA:

        y = W0 x + (alpha/r) B A x

    W0 is frozen.
    A and B are trainable.
    """
    def __init__(
        self,
        layer,
        rank=16,
        alpha=32,
        dropout=0.05,
    ):
        super().__init__()

        if not isinstance(layer, nn.Linear):
            raise TypeError("LoRALinear expects nn.Linear")

        self.layer = layer
        self.rank = int(rank)
        self.alpha = float(alpha)
        self.scaling = self.alpha / self.rank
        self.dropout = nn.Dropout(dropout)

        for p in self.layer.parameters():
            p.requires_grad_(False)

        self.A = nn.Parameter(
            torch.empty(
                self.rank,
                layer.in_features,
            )
        )
        self.B = nn.Parameter(
            torch.zeros(
                layer.out_features,
                self.rank,
            )
        )

        nn.init.kaiming_uniform_(
            self.A,
            a=5**0.5,
        )

    def forward(self, x):
        base_output = self.layer(x)

        low_rank = F.linear(
            self.dropout(x),
            self.A,
        )

        update = F.linear(
            low_rank,
            self.B,
        )

        return base_output + self.scaling * update


In [16]:
# ============================================================
# Build TEACHER from the supplied best.pt
# Then build a 6-layer STUDENT from teacher's first 6 layers.
# LoRA is retained on the four attention projections.
# ============================================================

def strip_module_prefix(state_dict):
    return {
        (k[7:] if k.startswith("module.") else k): v
        for k, v in state_dict.items()
    }


def extract_model_state_dict(checkpoint):
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        return checkpoint["model_state_dict"]
    return checkpoint


def build_base_model(num_layers):
    cfg = LMConfig(
        vocab_size=VOCAB_SIZE,
        d_model=512,
        n_layers=num_layers,
        n_heads=8,
        n_kv_heads=8,
        d_ff=4096,
        max_seq_len=MAX_SEQ_LEN,
        dropout=0.1,
        pos_emb_type="sinusoidal",
        feat_dim=32,
        chunk=128,
    )
    return DecoderLM(cfg)


def inject_lora(model):
    target_names = (
        "q_proj",
        "k_proj",
        "v_proj",
        "out_proj",
    )

    for block in model.blocks:
        for name in target_names:
            old_layer = getattr(block.attn, name)
            setattr(
                block.attn,
                name,
                LoRALinear(
                    old_layer,
                    rank=LORA_RANK,
                    alpha=LORA_ALPHA,
                    dropout=LORA_DROPOUT,
                ),
            )
    return model


# ------------------------------------------------------------
# 1. Load best.pt into the 12-layer TEACHER.
# ------------------------------------------------------------
teacher_model = build_base_model(TEACHER_N_LAYERS)

if not os.path.exists(TEACHER_CHECKPOINT_PATH):
    raise FileNotFoundError(
        f"Teacher checkpoint not found: {TEACHER_CHECKPOINT_PATH}"
    )

teacher_checkpoint = torch.load(
    TEACHER_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

teacher_state_dict = strip_module_prefix(
    extract_model_state_dict(teacher_checkpoint)
)

teacher_model = inject_lora(teacher_model)

missing, unexpected = teacher_model.load_state_dict(
    teacher_state_dict,
    strict=True,
)

if missing or unexpected:
    raise RuntimeError(
        "Teacher checkpoint does not exactly match the "
        "12-layer LinearAttention + LoRA architecture.\n"
        f"Missing: {missing}\n"
        f"Unexpected: {unexpected}"
    )

teacher_model = teacher_model.to(device)
teacher_model.eval()

for p in teacher_model.parameters():
    p.requires_grad_(False)

print("Teacher loaded successfully.")
print("Teacher checkpoint:", TEACHER_CHECKPOINT_PATH)
print("Teacher epoch:", teacher_checkpoint.get("epoch"))
print("Teacher metrics:", teacher_checkpoint.get("metrics"))


# ------------------------------------------------------------
# 2. Build 6-layer STUDENT.
# ------------------------------------------------------------
student_model = build_base_model(STUDENT_N_LAYERS)
student_model = inject_lora(student_model)

teacher_state = teacher_model.state_dict()
student_state = student_model.state_dict()

# Copy embeddings and final projection/norm where dimensions match.
copy_keys = [
    "tok_emb.weight",
    "ln_f.weight",
    "ln_f.bias",
    "head.weight",
]

for key in copy_keys:
    if key in teacher_state and key in student_state:
        if teacher_state[key].shape == student_state[key].shape:
            student_state[key] = teacher_state[key].detach().cpu().clone()

# Copy the first six teacher blocks exactly into the student.
for layer_idx in range(STUDENT_N_LAYERS):
    prefix = f"blocks.{layer_idx}."
    for key in list(student_state.keys()):
        if key.startswith(prefix) and key in teacher_state:
            if student_state[key].shape != teacher_state[key].shape:
                raise RuntimeError(
                    f"Shape mismatch while copying {key}: "
                    f"student={tuple(student_state[key].shape)}, "
                    f"teacher={tuple(teacher_state[key].shape)}"
                )
            student_state[key] = teacher_state[key].detach().cpu().clone()

student_model.load_state_dict(
    student_state,
    strict=True,
)

student_model = student_model.to(device)

# Only LoRA A/B are trainable in the student.
for name, p in student_model.named_parameters():
    p.requires_grad_(name.endswith(".A") or name.endswith(".B"))

student_trainable = sum(
    p.numel() for p in student_model.parameters() if p.requires_grad
)
student_total = sum(
    p.numel() for p in student_model.parameters()
)

print("\nStudent initialized from teacher's first 6 layers.")
print(f"Student trainable parameters: {student_trainable:,}")
print(f"Student total parameters:     {student_total:,}")
print(f"Student trainable %:          {100 * student_trainable / student_total:.4f}%")

# Safety checks.
assert all(not p.requires_grad for p in teacher_model.parameters())
assert any(p.requires_grad for p in student_model.parameters())
assert all(
    (p.requires_grad == (name.endswith(".A") or name.endswith(".B")))
    for name, p in student_model.named_parameters()
)

clear_cuda_memory()


Teacher loaded successfully.
Teacher checkpoint: /kaggle/input/datasets/rogue2222/kerenel-p7/PCR_LinearAttention_LoRA_Random_Negatives/best.pt
Teacher epoch: 9
Teacher metrics: {'Micro-Precision@7': 0.26150121065375304, 'Micro-Recall@7': 0.22199383350462487, 'Micro-F1@7': 0.24013340744858255}

Student initialized from teacher's first 6 layers.
Student trainable parameters: 393,216
Student total parameters:     36,013,184
Student trainable %:          1.0919%


In [17]:
# -------------------------
# Student LoRA optimizer
# -------------------------
lora_a_parameters = []
lora_b_parameters = []

for name, p in student_model.named_parameters():
    if not p.requires_grad:
        continue

    if name.endswith(".A"):
        lora_a_parameters.append(p)
    elif name.endswith(".B"):
        lora_b_parameters.append(p)

optimizer = torch.optim.AdamW(
    [
        {
            "params": lora_a_parameters,
            "lr": LORA_A_LR,
            "weight_decay": WEIGHT_DECAY,
        },
        {
            "params": lora_b_parameters,
            "lr": LORA_B_LR,
            "weight_decay": WEIGHT_DECAY,
        },
    ],
    betas=(0.9, 0.999),
    eps=1e-8,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=torch.cuda.is_available() and USE_AMP,
)

print("LoRA A tensors:", len(lora_a_parameters))
print("LoRA B tensors:", len(lora_b_parameters))
print("A LR:", LORA_A_LR)
print("B LR:", LORA_B_LR)


LoRA A tensors: 24
LoRA B tensors: 24
A LR: 2e-05
B LR: 2e-05


In [18]:
# ============================================================
# Encode documents
# ============================================================

def mean_pool_tokens(
    hidden,
    valid_mask,
):
    """
    valid_mask=True means VALID token.
    """
    mask = valid_mask.unsqueeze(
        -1
    ).to(
        hidden.dtype
    )

    return (
        (hidden * mask).sum(
            dim=1
        )
        / mask.sum(
            dim=1
        ).clamp_min(1.0)
    )


def encode_query_tensor(
    model,
    query_chunks,
):
    """
    Encode one document.

    Important mask conventions:
      * token_valid_mask=True  -> VALID
      * attention padding_mask=True -> PAD
        (required by chunked_gqa_linear_attn)
    """
    chunks = query_chunks.to(
        device,
        non_blocking=True,
    )

    valid = (
        chunks != PAD_ID
    ).any(dim=1)

    valid_chunks = chunks[
        valid
    ]

    if valid_chunks.size(0) == 0:
        return F.normalize(
            torch.zeros(
                model.cfg.d_model,
                device=device,
            ),
            p=2,
            dim=-1,
        )

    chunk_embs = []

    for j in range(
        0,
        valid_chunks.size(0),
        CHUNK_BATCH_SIZE,
    ):
        token_ids = valid_chunks[
            j:j + CHUNK_BATCH_SIZE
        ]

        # Pooling needs TRUE for valid tokens.
        token_valid_mask = (
            token_ids != PAD_ID
        )

        # Supplied LinearAttention kernel needs TRUE for PAD.
        attention_padding_mask = (
            token_ids == PAD_ID
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(
                USE_AMP
                and torch.cuda.is_available()
            ),
        ):
            hidden = model.encode(
                token_ids,
                padding_mask=attention_padding_mask,
            )

            pooled = mean_pool_tokens(
                hidden,
                token_valid_mask,
            )

        chunk_embs.append(
            pooled.float()
        )

        del (
            token_ids,
            token_valid_mask,
            attention_padding_mask,
            hidden,
            pooled,
        )

    chunk_embs = torch.cat(
        chunk_embs,
        dim=0,
    )

    document_emb = chunk_embs.mean(
        dim=0
    )

    return F.normalize(
        document_emb.float(),
        p=2,
        dim=-1,
    )


@torch.no_grad()
def encode_documents(
    model,
    records,
    document_ids,
    device,
    chunk_batch_size=1,
):
    model.eval()

    document_embeddings = []

    for doc_start in tqdm(
        range(
            len(document_ids)
        ),
        desc="Encoding documents",
    ):
        doc_id = document_ids[
            doc_start
        ]

        chunks = records[
            doc_id
        ]["chunks"]

        emb = encode_query_tensor(
            model,
            chunks,
        )

        document_embeddings.append(
            emb.cpu()
        )

        del chunks, emb

        if (
            torch.cuda.is_available()
            and doc_start % 25 == 0
        ):
            torch.cuda.empty_cache()

    return torch.stack(
        document_embeddings,
        dim=0,
    )


In [19]:
# ============================================================
# KD SANITY CHECKS
# ============================================================

teacher_model.eval()
student_model.eval()

sample_qid = next(iter(train_query_records))
sample_chunks = train_query_records[sample_qid]["chunks"].to(device)

with torch.no_grad():
    teacher_emb = encode_query_tensor(
        teacher_model,
        sample_chunks,
    )
    student_emb = encode_query_tensor(
        student_model,
        sample_chunks,
    )

assert teacher_emb.shape == student_emb.shape
assert teacher_emb.ndim == 1
assert torch.isfinite(teacher_emb).all()
assert torch.isfinite(student_emb).all()

print("KD smoke test passed.")
print("Embedding shape:", tuple(student_emb.shape))
print("Teacher norm:", float(teacher_emb.norm().item()))
print("Student norm:", float(student_emb.norm().item()))

student_model.train()
teacher_model.eval()

print("Teacher requires_grad:", any(p.requires_grad for p in teacher_model.parameters()))
print("Student trainable parameters:",
      sum(p.numel() for p in student_model.parameters() if p.requires_grad))


KD smoke test passed.
Embedding shape: (512,)
Teacher norm: 1.0
Student norm: 1.0
Teacher requires_grad: False
Student trainable parameters: 393216


In [20]:
# -------------------------
# Multi-positive contrastive loss
# -------------------------
def multi_positive_contrastive_loss(
    similarity_scores,
    positive_mask,
    candidate_mask,
    temperature=0.07,
):
    logits = similarity_scores / temperature

    logits = logits.masked_fill(
        ~candidate_mask,
        float("-inf"),
    )

    positive_logits = logits.masked_fill(
        ~positive_mask,
        float("-inf"),
    )

    valid_rows = (
        positive_mask.any(dim=1)
        & candidate_mask.any(dim=1)
    )

    if not valid_rows.all():
        raise ValueError(
            "Every query must have at least one positive "
            "and one candidate."
        )

    numerator = torch.logsumexp(
        positive_logits,
        dim=1,
    )

    denominator = torch.logsumexp(
        logits,
        dim=1,
    )

    return -(numerator - denominator).mean()


In [21]:
# ============================================================
# One training epoch with retrieval loss + KD loss
# ============================================================

def knowledge_distillation_loss(
    student_embedding,
    teacher_embedding,
):
    student_embedding = F.normalize(
        student_embedding.float(),
        p=2,
        dim=-1,
    )
    teacher_embedding = F.normalize(
        teacher_embedding.float(),
        p=2,
        dim=-1,
    )

    return F.mse_loss(
        student_embedding,
        teacher_embedding,
    )


@torch.no_grad()
def encode_teacher(
    teacher,
    chunks,
):
    teacher.eval()
    return encode_query_tensor(
        teacher,
        chunks,
    )


def train_one_epoch_kd(
    student,
    teacher,
    train_loader,
    optimizer,
    scaler,
    device,
):
    student.train()
    teacher.eval()

    running_total = 0.0
    running_retrieval = 0.0
    running_kd = 0.0
    count = 0

    for batch in tqdm(
        train_loader,
        desc="Training KD",
        leave=False,
    ):
        samples = batch if isinstance(batch, list) else [batch]
        batch_size_actual = len(samples)

        optimizer.zero_grad(set_to_none=True)

        batch_total = 0.0
        batch_retrieval = 0.0
        batch_kd = 0.0

        for sample in samples:
            query_chunks = sample["query_chunks"].to(
                device, non_blocking=True
            )
            candidate_chunks = sample["candidate_chunks"].to(
                device, non_blocking=True
            )
            positive_mask = sample["positive_mask"].to(
                device, non_blocking=True
            )

            # ------------------------------------------------
            # STUDENT: gradients enabled
            # ------------------------------------------------
            student_query_embedding = encode_query_tensor(
                student,
                query_chunks,
            )

            student_candidate_embeddings = []
            for c in range(candidate_chunks.size(0)):
                student_candidate_embeddings.append(
                    encode_query_tensor(
                        student,
                        candidate_chunks[c],
                    )
                )

            student_candidate_embeddings = torch.stack(
                student_candidate_embeddings,
                dim=0,
            )

            similarities = (
                student_query_embedding.unsqueeze(0)
                @ student_candidate_embeddings.transpose(0, 1)
            )

            retrieval_loss = multi_positive_contrastive_loss(
                similarities,
                positive_mask.unsqueeze(0),
                torch.ones_like(
                    positive_mask,
                    dtype=torch.bool,
                ).unsqueeze(0),
                temperature=TEMPERATURE,
            )

            # ------------------------------------------------
            # TEACHER: frozen, no gradients
            # ------------------------------------------------
            with torch.no_grad():
                teacher_query_embedding = encode_teacher(
                    teacher,
                    query_chunks,
                )

                teacher_candidate_embeddings = []
                for c in range(candidate_chunks.size(0)):
                    teacher_candidate_embeddings.append(
                        encode_teacher(
                            teacher,
                            candidate_chunks[c],
                        )
                    )

                teacher_candidate_embeddings = torch.stack(
                    teacher_candidate_embeddings,
                    dim=0,
                )

            # ------------------------------------------------
            # KD on query + every candidate representation.
            # Embeddings are normalized inside KD loss.
            # ------------------------------------------------
            kd_query_loss = knowledge_distillation_loss(
                student_query_embedding,
                teacher_query_embedding,
            )

            kd_candidate_loss = knowledge_distillation_loss(
                student_candidate_embeddings,
                teacher_candidate_embeddings,
            )

            kd_loss = (
                kd_query_loss + kd_candidate_loss
            ) / 2.0

            total_loss = (
                retrieval_loss
                + KD_WEIGHT * kd_loss
            )

            # Accumulate gradients over the two queries,
            # while processing one query graph at a time.
            loss_for_backward = (
                total_loss / batch_size_actual
            )

            if USE_AMP and torch.cuda.is_available():
                scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

            batch_total += float(total_loss.detach().item())
            batch_retrieval += float(retrieval_loss.detach().item())
            batch_kd += float(kd_loss.detach().item())

            del (
                query_chunks,
                candidate_chunks,
                positive_mask,
                student_query_embedding,
                student_candidate_embeddings,
                teacher_query_embedding,
                teacher_candidate_embeddings,
                similarities,
                retrieval_loss,
                kd_query_loss,
                kd_candidate_loss,
                kd_loss,
                total_loss,
                loss_for_backward,
            )

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        # ----------------------------------------------------
        # One optimizer update for the full batch.
        # ----------------------------------------------------
        if USE_AMP and torch.cuda.is_available():
            scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            [
                p for p in student.parameters()
                if p.requires_grad
            ],
            GRAD_CLIP,
        )

        if USE_AMP and torch.cuda.is_available():
            scaler.step(optimizer)
            scaler.update()
        else:
            optimizer.step()

        batch_total /= batch_size_actual
        batch_retrieval /= batch_size_actual
        batch_kd /= batch_size_actual

        running_total += batch_total
        running_retrieval += batch_retrieval
        running_kd += batch_kd
        count += 1

        del samples

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return {
        "total_loss": running_total / max(count, 1),
        "retrieval_loss": running_retrieval / max(count, 1),
        "kd_loss": running_kd / max(count, 1),
    }


In [22]:
# -------------------------
# Full-pool retrieval evaluation
# -------------------------
@torch.no_grad()
def evaluate_retrieval_full_pool(
    model,
    query_records,
    candidate_records,
    query_data,
    k=7,
    eval_batch_size=1,
):
    model.eval()

    candidate_ids = list(
        candidate_records.keys()
    )

    query_ids = [
        qid
        for qid in query_data.keys()
        if query_data[qid].get(
            "relevant_candidates"
        )
    ]

    candidate_embeddings = (
        encode_documents(
            model,
            candidate_records,
            candidate_ids,
            device,
            chunk_batch_size=1,
        )
    )

    query_embeddings = []

    for qid in tqdm(
        query_ids,
        desc="Encoding queries",
    ):
        emb = encode_query_tensor(
            model,
            query_records[qid]["chunks"],
        )
        query_embeddings.append(
            emb.cpu()
        )

    query_embeddings = torch.stack(
        query_embeddings
    )

    hits = 0
    total_relevant = 0
    total_retrieved = (
        len(query_ids) * k
    )

    predictions = {}

    for i, qid in enumerate(
        query_ids
    ):
        scores = (
            query_embeddings[
                i:i + 1
            ]
            @ candidate_embeddings.T
        ).squeeze(0)

        topk = min(
            k,
            scores.numel(),
        )

        values, indices = torch.topk(
            scores,
            k=topk,
        )

        retrieved = [
            candidate_ids[int(j)]
            for j in indices
        ]

        relevant = set(
            query_data[qid][
                "relevant_candidates"
            ]
        )

        intersection = len(
            set(retrieved)
            & relevant
        )

        hits += intersection
        total_relevant += len(
            relevant
        )

        predictions[qid] = {
            "retrieved_candidates": retrieved,
            "relevant_candidates": list(
                relevant
            ),
            "scores": [
                float(v)
                for v in values
            ],
        }

    precision = (
        hits / total_retrieved
        if total_retrieved
        else 0.0
    )

    recall = (
        hits / total_relevant
        if total_relevant
        else 0.0
    )

    f1 = (
        2 * precision * recall
        / (precision + recall)
        if precision + recall > 0
        else 0.0
    )

    return {
        "Micro-Precision@7": precision,
        "Micro-Recall@7": recall,
        "Micro-F1@7": f1,
    }, predictions


In [23]:
# -------------------------
# Student KD checkpoint helper
# -------------------------
def save_checkpoint(
    path,
    epoch,
    student,
    optimizer,
    scaler,
    train_losses,
    metrics=None,
):
    payload = {
        "epoch": epoch,
        "model_state_dict": student.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "train_losses": train_losses,
        "metrics": metrics,
        "config": {
            "max_seq_len": MAX_SEQ_LEN,
            "chunk_overlap": CHUNK_OVERLAP,
            "max_chunks": MAX_CHUNKS,
            "num_negatives": NUM_NEGATIVES,
            "chunk_batch_size": CHUNK_BATCH_SIZE,
            "batch_size": BATCH_SIZE,
            "temperature": TEMPERATURE,
            "kd_weight": KD_WEIGHT,
            "teacher_layers": TEACHER_N_LAYERS,
            "student_layers": STUDENT_N_LAYERS,
            "lora_rank": LORA_RANK,
            "lora_alpha": LORA_ALPHA,
            "lora_dropout": LORA_DROPOUT,
            "lora_a_lr": LORA_A_LR,
            "lora_b_lr": LORA_B_LR,
            "adaptation": "standard additive LoRA + KD + random negatives",
            "backbone": "decoder-only GQA LinearAttention",
            "teacher_checkpoint": TEACHER_CHECKPOINT_PATH,
        },
    }

    torch.save(payload, path)
    return path


In [24]:
# ============================================================
# TRAIN STUDENT WITH KD
# ============================================================

best_f1 = -1.0
history = []

for epoch in range(1, EPOCHS + 1):
    print(f"\n===== KD Epoch {epoch}/{EPOCHS} =====")

    clear_cuda_memory()
    gpu_memory_report("Before train | ")

    train_losses = train_one_epoch_kd(
        student=student_model,
        teacher=teacher_model,
        train_loader=train_loader,
        optimizer=optimizer,
        scaler=scaler,
        device=device,
    )

    gpu_memory_report("After train | ")

    epoch_checkpoint = os.path.join(
        CHECKPOINT_DIR,
        f"student_epoch_{epoch}.pt",
    )

    save_checkpoint(
        epoch_checkpoint,
        epoch,
        student_model,
        optimizer,
        scaler,
        train_losses,
        metrics=None,
    )

    # Official full-pool dev retrieval is always performed
    # with the STUDENT, not the teacher.
    dev_metrics, dev_predictions = (
        evaluate_retrieval_full_pool(
            model=student_model,
            query_records=dev_query_records,
            candidate_records=dev_candidate_records,
            query_data=dev_data,
            k=7,
            eval_batch_size=1,
        )
    )

    row = {
        "epoch": epoch,
        **train_losses,
        **dev_metrics,
    }

    history.append(row)

    with open(HISTORY_PATH, "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2)

    print(f"Total train loss: {train_losses['total_loss']:.6f}")
    print(f"Retrieval loss:   {train_losses['retrieval_loss']:.6f}")
    print(f"KD loss:          {train_losses['kd_loss']:.6f}")
    print(f"Precision@7:      {dev_metrics['Micro-Precision@7']:.6f}")
    print(f"Recall@7:         {dev_metrics['Micro-Recall@7']:.6f}")
    print(f"F1@7:             {dev_metrics['Micro-F1@7']:.6f}")

    current_f1 = dev_metrics["Micro-F1@7"]

    if current_f1 > best_f1:
        best_f1 = current_f1

        save_checkpoint(
            BEST_CHECKPOINT,
            epoch,
            student_model,
            optimizer,
            scaler,
            train_losses,
            metrics=dev_metrics,
        )

        with open(
            os.path.join(
                CHECKPOINT_DIR,
                "best_dev_predictions.json",
            ),
            "w",
            encoding="utf-8",
        ) as f:
            json.dump(dev_predictions, f, indent=2)

        print("Saved new best student checkpoint:", BEST_CHECKPOINT)

    clear_cuda_memory()

print("\nBest Student Dev F1@7:", best_f1)



===== KD Epoch 1/10 =====
Before train | GPU 0: allocated=0.40 GB, reserved=0.43 GB, free=14.00 GB / total=14.56 GB
Before train | GPU 1: allocated=0.00 GB, reserved=0.00 GB, free=14.46 GB / total=14.56 GB


Training KD:   0%|          | 0/103 [00:00<?, ?it/s]

After train | GPU 0: allocated=0.41 GB, reserved=0.46 GB, free=13.96 GB / total=14.56 GB
After train | GPU 1: allocated=0.00 GB, reserved=0.00 GB, free=14.46 GB / total=14.56 GB


Encoding documents:   0%|          | 0/1023 [00:00<?, ?it/s]

Encoding queries:   0%|          | 0/118 [00:00<?, ?it/s]

Total train loss: 0.728165
Retrieval loss:   0.726583
KD loss:          0.003164
Precision@7:      0.083535
Recall@7:         0.070915
F1@7:             0.076709
Saved new best student checkpoint: /kaggle/working/PCR_LinearAttention_LoRA_KD_Random_Negatives/best_student.pt

===== KD Epoch 2/10 =====
Before train | GPU 0: allocated=0.41 GB, reserved=0.46 GB, free=13.96 GB / total=14.56 GB
Before train | GPU 1: allocated=0.00 GB, reserved=0.00 GB, free=14.46 GB / total=14.56 GB


... (41 outputs left out)

After train | GPU 0: allocated=0.41 GB, reserved=0.46 GB, free=13.96 GB / total=14.56 GB
After train | GPU 1: allocated=0.00 GB, reserved=0.00 GB, free=14.46 GB / total=14.56 GB


Encoding documents:   0%|          | 0/1023 [00:00<?, ?it/s]

Encoding queries:   0%|          | 0/118 [00:00<?, ?it/s]

Total train loss: 0.340383
Retrieval loss:   0.338720
KD loss:          0.003326
Precision@7:      0.110169
Recall@7:         0.093525
F1@7:             0.101167
Saved new best student checkpoint: /kaggle/working/PCR_LinearAttention_LoRA_KD_Random_Negatives/best_student.pt

Best Student Dev F1@7: 0.10116731517509726


In [25]:
# ============================================================
# BEST STUDENT CHECKPOINT PATH
# ============================================================
BEST_STUDENT_CHECKPOINT = BEST_CHECKPOINT
print("Best student checkpoint:", BEST_STUDENT_CHECKPOINT)


Best student checkpoint: /kaggle/working/PCR_LinearAttention_LoRA_KD_Random_Negatives/best_student.pt


In [26]:
print(json.dumps(history, indent=2))
print("Best Dev F1@7:", best_f1)
print("Best student checkpoint:", BEST_STUDENT_CHECKPOINT)


[
  {
    "epoch": 1,
    "total_loss": 0.7281649630806111,
    "retrieval_loss": 0.7265829496013309,
    "kd_loss": 0.0031640273917272,
    "Micro-Precision@7": 0.08353510895883777,
    "Micro-Recall@7": 0.07091469681397738,
    "Micro-F1@7": 0.07670928293496387
  },
  {
    "epoch": 2,
    "total_loss": 0.6263985759081337,
    "retrieval_loss": 0.6248245366568704,
    "kd_loss": 0.0031480811944895977,
    "Micro-Precision@7": 0.08232445520581114,
    "Micro-Recall@7": 0.0698869475847893,
    "Micro-F1@7": 0.07559755419677598
  },
  {
    "epoch": 3,
    "total_loss": 0.4868720813074547,
    "retrieval_loss": 0.4853140798587244,
    "kd_loss": 0.0031160028488601294,
    "Micro-Precision@7": 0.08837772397094432,
    "Micro-Recall@7": 0.0750256937307297,
    "Micro-F1@7": 0.0811561978877154
  },
  {
    "epoch": 4,
    "total_loss": 0.43713954336731015,
    "retrieval_loss": 0.4355818808657452,
    "kd_loss": 0.0031153241297325065,
    "Micro-Precision@7": 0.09200968523002422,
    "Micr

In [27]:


def load_best_student_checkpoint(
    model,
    checkpoint_path,
):
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(
            f"Best student checkpoint not found: {checkpoint_path}"
        )

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    state_dict = strip_module_prefix(
        extract_model_state_dict(checkpoint)
    )

    missing, unexpected = model.load_state_dict(
        state_dict,
        strict=True,
    )

    if missing or unexpected:
        raise RuntimeError(
            "Student checkpoint mismatch.\n"
            f"Missing: {missing}\n"
            f"Unexpected: {unexpected}"
        )

    model = model.to(device)
    model.eval()

    print("Loaded best student checkpoint:", checkpoint_path)
    print("Epoch:", checkpoint.get("epoch"))
    print("Dev metrics:", checkpoint.get("metrics"))

    return checkpoint


@torch.no_grad()
def generate_test_retrieval(
    model,
    query_records,
    candidate_records,
    test_data,
    top_k=7,
):
    model.eval()

    candidate_ids = list(candidate_records.keys())

    candidate_embeddings = encode_documents(
        model,
        candidate_records,
        candidate_ids,
        device,
        chunk_batch_size=1,
    )

    predictions = {}

    for qid in tqdm(
        list(test_data.keys()),
        desc="Generating test retrieval",
    ):
        qid = str(qid)

        query_embedding = encode_query_tensor(
            model,
            query_records[qid]["chunks"],
        ).cpu()

        scores = (
            query_embedding.unsqueeze(0)
            @ candidate_embeddings.T
        ).squeeze(0)

        valid_candidate_indices = [
            i for i, cid in enumerate(candidate_ids)
            if str(cid) != qid
        ]

        if len(valid_candidate_indices) < top_k:
            raise RuntimeError(
                f"Query {qid} has only "
                f"{len(valid_candidate_indices)} valid candidates."
            )

        valid_indices = torch.tensor(
            valid_candidate_indices,
            dtype=torch.long,
            device=scores.device,
        )

        valid_scores = scores[valid_indices]

        values, local_indices = torch.topk(
            valid_scores,
            k=top_k,
        )

        retrieved = [
            str(candidate_ids[int(valid_indices[int(i)])])
            for i in local_indices
        ]

        if len(retrieved) != 7:
            raise RuntimeError(
                f"Query {qid} produced {len(retrieved)} candidates."
            )

        if qid in retrieved:
            raise RuntimeError(
                f"Query {qid} appears in its own predictions."
            )

        if len(set(retrieved)) != 7:
            raise RuntimeError(
                f"Query {qid} has duplicate candidates."
            )

        predictions[qid] = retrieved

        del (
            query_embedding,
            scores,
            valid_scores,
            valid_indices,
            values,
            local_indices,
        )

    return predictions


clear_cuda_memory()

best_student_checkpoint = load_best_student_checkpoint(
    student_model,
    BEST_STUDENT_CHECKPOINT,
)

test_predictions = generate_test_retrieval(
    model=student_model,
    query_records=test_query_records,
    candidate_records=test_candidate_records,
    test_data=test_data,
    top_k=7,
)

print("\nValidating predictions...")

for qid, retrieved in test_predictions.items():
    assert isinstance(retrieved, list)
    assert len(retrieved) == 7
    assert qid not in retrieved
    assert len(set(retrieved)) == 7
    assert all(
        isinstance(cid, str) and cid.strip()
        for cid in retrieved
    )

print(
    f"Validation passed for {len(test_predictions)} queries."
)

PREDICTION_FILE = (
    f"{CHECKPOINT_DIR}/prediction.json"
)

with open(
    PREDICTION_FILE,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        test_predictions,
        f,
        indent=2,
    )

print("Saved Codabench prediction file:")
print(PREDICTION_FILE)

first_qid = next(iter(test_predictions))
print("\nExample:")
print(
    json.dumps(
        {first_qid: test_predictions[first_qid]},
        indent=2,
    )
)


Loaded best student checkpoint: /kaggle/working/PCR_LinearAttention_LoRA_KD_Random_Negatives/best_student.pt
Epoch: 10
Dev metrics: {'Micro-Precision@7': 0.11016949152542373, 'Micro-Recall@7': 0.09352517985611511, 'Micro-F1@7': 0.10116731517509726}


Encoding documents:   0%|          | 0/1727 [00:00<?, ?it/s]

Generating test retrieval:   0%|          | 0/237 [00:00<?, ?it/s]


Validating predictions...
Validation passed for 237 queries.
Saved Codabench prediction file:
/kaggle/working/PCR_LinearAttention_LoRA_KD_Random_Negatives/prediction.json

Example:
{
  "0000021652": [
    "0001102670",
    "0000916840",
    "0001505859",
    "0001037935",
    "0001813941",
    "0000307495",
    "0000770422"
  ]
}


In [28]:
import zipfile

ZIP_FILE = (
    f"{CHECKPOINT_DIR}/prediction.zip"
)

with zipfile.ZipFile(
    ZIP_FILE,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zipf:
    zipf.write(
        PREDICTION_FILE,
        arcname="prediction.json",
    )

print("Created:", ZIP_FILE)

with zipfile.ZipFile(ZIP_FILE, "r") as zipf:
    print("ZIP contents:", zipf.namelist())


Created: /kaggle/working/PCR_LinearAttention_LoRA_KD_Random_Negatives/prediction.zip
ZIP contents: ['prediction.json']
